# DS 227 · Week 10 — Data storytelling
## Narrative, dashboards and communication

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 7 — Charts on these slides draw in the browser


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 17 — One dashboard, one question


**Exercise.**

1. Predict the order of the months in the printed table.
2. Run the cell.
3. Delete the reindex line and run again. In what order are the months now, and why?


In [ ]:
sales = pd.DataFrame({
    "month":  ["Jan", "Feb", "Mar", "Apr", "May", "Jun"] * 2,
    "region": ["Visayas"] * 6 + ["Luzon"] * 6,
    "sales":  [10, 12, 11, 14, 16, 18, 20, 19, 22, 21, 25, 27],
})
wide = sales.pivot_table(index="month", columns="region", values="sales")
wide = wide.reindex(["Jan", "Feb", "Mar", "Apr", "May", "Jun"])
print(wide)

### Slide 21 — Small multiples: a grid of one-point charts


In [ ]:
# sales, wide: the monthly sales table (previous slides)
totals = sales.groupby("region")["sales"].sum()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
wide.plot(ax=a1, marker="o")
a1.set_title("Trend by region")
totals.plot.bar(ax=a2)
a2.set_title("Total by region")
plt.tight_layout()
plt.show()

### Slide 25 — Layout directs the reader's scan


**Exercise.**

1. Predict the order of the three bars.
2. Run the cell.
3. Remove .sort_values(ascending=False) and run again. What must the reader now do?


In [ ]:
import matplotlib.pyplot as plt

totals = pd.Series({"Luzon": 134, "Visayas": 81, "Mindanao": 108})   # the lab's totals

fig, ax = plt.subplots()
totals.sort_values(ascending=False).plot.bar(ax=ax)
plt.show()

### Slide 30 — The title states the finding, not the variable


**Exercise.**

1. Check the title’s 35% against the data before running.
2. Run the cell.
3. Visayas went from 10 to 18. Write and run its finding-title.


In [ ]:
import matplotlib.pyplot as plt

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
luzon  = [20, 19, 22, 21, 25, 27]

fig, ax = plt.subplots()
ax.plot(months, luzon, marker="o")
ax.set_title("Luzon sales grew 35% over the half-year")   # not "Sales"
plt.show()

### Slide 34 — Removing elements that do not carry the message


In [ ]:
# wide: the monthly sales table
fig, ax = plt.subplots()
wide.plot(ax=ax, marker="o")                     # the default
plt.show()

fig, ax = plt.subplots()
wide.plot(ax=ax, marker="o")
ax.spines[["top", "right"]].set_visible(False)   # the four lines
ax.grid(axis="y", alpha=0.3)
ax.tick_params(length=0)
ax.legend(frameon=False)
plt.show()

### Slide 36 — The flood data, loaded from the course API


Expected output:

```
(34079, 15)
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
df   = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text))

print(df.shape)

### Slide 37 — Unusable precision is noise


Expected output:

```
1600088782392.83
PHP 1.60T
```


In [ ]:
# in the notebook: full precision
total = df.budget.sum()
print(total)

# on the chart: human scale
print(f"PHP {total / 1e12:.2f}T")

### Slide 39 — Writing the conclusion on the chart


In [ ]:
c = df.groupby("contractor").budget.sum().sort_values(ascending=False)
share = c.cumsum() / c.sum()

fig, ax = plt.subplots()
ax.plot(range(1, len(share) + 1), share.values)
ax.annotate("48 firms = 27.6% of value",
            xy=(48, 0.276), xytext=(400, 0.45),
            arrowprops=dict(arrowstyle="->", color="#555"))
ax.axhline(0.276, ls="--", c="#999", lw=1)    # a reference line
plt.show()

### Slide 46 — Dashboards repeat errors as well


Expected output:

```
year
2024    5553
2025    3475
2026       1
dtype: int64
2026 is partial: 1 project(s)
```


In [ ]:
counts = df.groupby("year").size()
print(counts.tail(3))

# guard the incomplete period before charting
latest = counts.index.max()
if counts[latest] < counts.median():
    print(f"{latest} is partial: {counts[latest]} project(s)")

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
